<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_basic/05_rnn_lstm_basics.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# RNN/LSTM 기초 - 순환 신경망과 텍스트 처리

RNN(Recurrent Neural Network)과 LSTM(Long Short-Term Memory)은 시퀀스 데이터 처리에 특화된 딥러닝 모델입니다. 이 노트북에서는 RNN의 기본 개념부터 LSTM, 그리고 실제 텍스트 감성 분석까지 단계별로 학습합니다.

## 학습 목표
1. 순환 신경망(RNN)의 기본 개념과 구조 이해
2. LSTM과 GRU의 차이점과 장점 파악
3. 텍스트 전처리 및 시퀀스 처리 방법 학습
4. PyTorch를 사용한 RNN 모델 구현
5. 실제 데이터로 감성 분석 실습

In [ ]:
# PyTorch 임포트 (Google Colab T4 환경)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from collections import Counter, defaultdict
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

print(f"PyTorch 버전: {torch.__version__}")

# GPU 설정 (Google Colab T4 환경)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print(f"Using device: {device}")

# 시드 설정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. RNN의 기본 개념 이해하기

### 일반 신경망 vs RNN
- **일반 신경망**: 독립적인 입력 → 출력
- **RNN**: 이전 상태를 기억하여 시퀀스 처리

### RNN이 필요한 이유
1. **시간적 순서가 중요한 데이터**: 텍스트, 음성, 주가 등
2. **가변 길이 입력**: 문장마다 길이가 다름
3. **문맥 정보 활용**: 앞 단어들의 정보를 활용해 다음 단어 예측

In [ ]:
# RNN의 기본 구조와 개념 시각화
def visualize_rnn_concepts():
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. 일반 신경망 vs RNN
    ax1 = axes[0, 0]
    ax1.text(0.1, 0.8, '일반 신경망 (Feedforward)', fontsize=12, weight='bold')
    ax1.text(0.1, 0.6, '입력 → 처리 → 출력', fontsize=10)
    ax1.text(0.1, 0.5, '• 이전 입력 기억 안함', fontsize=9)
    ax1.text(0.1, 0.4, '• 고정 길이 입력', fontsize=9)
    
    ax1.text(0.1, 0.25, 'RNN (순환 신경망)', fontsize=12, weight='bold')
    ax1.text(0.1, 0.05, '입력 + 이전상태 → 처리 → 출력 + 현재상태', fontsize=10)
    ax1.text(0.1, -0.05, '• 이전 정보 기억', fontsize=9)
    ax1.text(0.1, -0.1, '• 가변 길이 입력', fontsize=9)
    
    ax1.set_xlim(0, 1)
    ax1.set_ylim(-0.15, 1)
    ax1.set_title('신경망 비교')
    ax1.axis('off')
    
    # 2. 시퀀스 처리 예시
    ax2 = axes[0, 1]
    sentence = ['안녕', '하세요', '좋은', '하루', '되세요']
    x_pos = np.arange(len(sentence))
    
    # 각 단어에 대한 막대 그래프
    colors = plt.cm.viridis(np.linspace(0, 1, len(sentence)))
    bars = ax2.bar(x_pos, [1]*len(sentence), color=colors, alpha=0.7)
    
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(sentence)
    ax2.set_title('시퀀스 데이터 예시')
    ax2.set_ylabel('입력 토큰')
    
    # 화살표로 순서 표시
    for i in range(len(sentence)-1):
        ax2.annotate('', xy=(i+0.4, 0.5), xytext=(i+0.6, 0.5),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
    
    # 3. RNN의 문제점 (기울기 소실)
    ax3 = axes[1, 0]
    time_steps = np.arange(1, 16)
    
    # 기울기 소실 시뮬레이션
    gradient_vanilla = np.exp(-time_steps * 0.2)  # 기울기 급격히 감소
    gradient_lstm = 0.8 + 0.2 * np.exp(-time_steps * 0.05)  # LSTM은 완만히 감소
    
    ax3.plot(time_steps, gradient_vanilla, 'r-o', label='기본 RNN', linewidth=2)
    ax3.plot(time_steps, gradient_lstm, 'b-s', label='LSTM', linewidth=2)
    ax3.set_xlabel('Time Steps (뒤로 갈수록)')
    ax3.set_ylabel('기울기 크기')
    ax3.set_title('기울기 소실 문제')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_yscale('log')
    
    # 4. RNN 계열 모델 비교
    ax4 = axes[1, 1]
    models = ['기본\nRNN', 'LSTM', 'GRU']
    complexity = [1, 4, 3]  # 복잡도
    performance = [2, 4.5, 4]  # 성능
    
    x = np.arange(len(models))
    width = 0.35
    
    bars1 = ax4.bar(x - width/2, complexity, width, label='복잡도', alpha=0.7, color='orange')
    bars2 = ax4.bar(x + width/2, performance, width, label='성능', alpha=0.7, color='green')
    
    ax4.set_xlabel('모델 타입')
    ax4.set_ylabel('상대적 점수')
    ax4.set_title('RNN 계열 모델 비교')
    ax4.set_xticks(x)
    ax4.set_xticklabels(models)
    ax4.legend()
    
    # 값 표시
    for bar in bars1:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{height}', ha='center', va='bottom')
    
    for bar in bars2:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{height}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

visualize_rnn_concepts()

## 2. 간단한 텍스트 데이터 생성 및 전처리

In [ ]:
# 감성 분석용 샘플 데이터 생성
positive_reviews = [
    "이 영화는 정말 훌륭합니다",
    "연기가 뛰어나고 스토리가 감동적입니다",
    "최고의 영화 중 하나입니다",
    "강력히 추천합니다",
    "완벽한 작품입니다",
    "매우 재미있고 흥미진진합니다",
    "놀라운 영화입니다",
    "감동적이고 아름다운 스토리입니다",
    "훌륭한 연출과 연기입니다",
    "이런 영화를 기다렸습니다",
    "excellent movie with great acting",
    "amazing story and wonderful characters",
    "highly recommend this masterpiece",
    "fantastic film with beautiful cinematography",
    "outstanding performance by all actors",
    "brilliant writing and direction",
    "loved every minute of this movie",
    "incredibly moving and inspiring",
    "best movie I have seen this year",
    "absolutely perfect in every way"
]

negative_reviews = [
    "이 영화는 정말 지루합니다",
    "스토리가 너무 뻔하고 예측 가능합니다",
    "시간 낭비였습니다",
    "연기가 어색하고 부자연스럽습니다",
    "실망스러운 작품입니다",
    "내용이 빈약하고 재미없습니다",
    "최악의 영화입니다",
    "돈이 아까운 영화입니다",
    "졸면서 봤습니다",
    "다시는 보고 싶지 않습니다",
    "terrible movie with poor acting",
    "boring plot and weak characters",
    "waste of time and money",
    "disappointing and predictable",
    "awful script and bad direction",
    "completely boring and uninteresting",
    "worst movie ever made",
    "fell asleep during the movie",
    "regret watching this film",
    "absolutely horrible in every aspect"
]

# 데이터와 라벨 결합
texts = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)  # 1: 긍정, 0: 부정

print(f"총 리뷰 개수: {len(texts)}")
print(f"긍정 리뷰: {len(positive_reviews)}개")
print(f"부정 리뷰: {len(negative_reviews)}개")

# 샘플 출력
print("\n샘플 데이터:")
for i in range(3):
    label_name = "긍정" if labels[i] == 1 else "부정"
    print(f"{label_name}: {texts[i]}")

print("\n부정 샘플:")
for i in range(len(positive_reviews), len(positive_reviews) + 3):
    label_name = "긍정" if labels[i] == 1 else "부정"
    print(f"{label_name}: {texts[i]}")

## 3. 텍스트 전처리 및 토큰화

In [ ]:
class TextPreprocessor:
    """
    텍스트 전처리를 위한 클래스
    """
    def __init__(self):
        self.vocab = {}
        self.word_to_idx = {}
        self.idx_to_word = {}
        self.vocab_size = 0
    
    def clean_text(self, text):
        """
        텍스트 정리 (소문자 변환, 특수문자 제거 등)
        """
        # 소문자 변환
        text = text.lower()
        # 특수문자 제거 (한글, 영문, 숫자, 공백만 유지)
        text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)
        # 여러 공백을 하나로
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def tokenize(self, text):
        """
        텍스트를 토큰으로 분할
        """
        return text.split()
    
    def build_vocab(self, texts):
        """
        어휘 사전 구축
        """
        word_counts = Counter()
        
        for text in texts:
            cleaned_text = self.clean_text(text)
            tokens = self.tokenize(cleaned_text)
            word_counts.update(tokens)
        
        # 특수 토큰 추가
        self.word_to_idx = {
            '<PAD>': 0,  # 패딩
            '<UNK>': 1,  # 미지의 단어
        }
        
        # 빈도순으로 단어 추가
        for word, count in word_counts.most_common():
            if count >= 1:  # 최소 1번 이상 등장한 단어만
                self.word_to_idx[word] = len(self.word_to_idx)
        
        # 역방향 사전
        self.idx_to_word = {idx: word for word, idx in self.word_to_idx.items()}
        self.vocab_size = len(self.word_to_idx)
        
        print(f"어휘 크기: {self.vocab_size}")
        print(f"가장 빈번한 10개 단어: {list(word_counts.most_common(10))}")
    
    def text_to_sequence(self, text):
        """
        텍스트를 숫자 시퀀스로 변환
        """
        cleaned_text = self.clean_text(text)
        tokens = self.tokenize(cleaned_text)
        sequence = []
        
        for token in tokens:
            if token in self.word_to_idx:
                sequence.append(self.word_to_idx[token])
            else:
                sequence.append(self.word_to_idx['<UNK>'])  # 미지의 단어
        
        return sequence
    
    def sequence_to_text(self, sequence):
        """
        숫자 시퀀스를 텍스트로 복원
        """
        words = []
        for idx in sequence:
            if idx in self.idx_to_word:
                words.append(self.idx_to_word[idx])
        return ' '.join(words)

# 전처리기 생성 및 어휘 구축
preprocessor = TextPreprocessor()
preprocessor.build_vocab(texts)

# 텍스트를 시퀀스로 변환
sequences = [preprocessor.text_to_sequence(text) for text in texts]

# 시퀀스 길이 분석
seq_lengths = [len(seq) for seq in sequences]

print(f"\n시퀀스 길이 통계:")
print(f"평균 길이: {np.mean(seq_lengths):.2f}")
print(f"최대 길이: {np.max(seq_lengths)}")
print(f"최소 길이: {np.min(seq_lengths)}")

# 예시 출력
print(f"\n변환 예시:")
sample_idx = 0
print(f"원본: {texts[sample_idx]}")
print(f"시퀀스: {sequences[sample_idx]}")
print(f"복원: {preprocessor.sequence_to_text(sequences[sample_idx])}")

## 4. 데이터 시각화 및 패딩

In [ ]:
# 데이터 탐색 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 시퀀스 길이 분포
ax1 = axes[0, 0]
ax1.hist(seq_lengths, bins=15, alpha=0.7, edgecolor='black')
ax1.axvline(np.mean(seq_lengths), color='red', linestyle='--', 
           label=f'평균: {np.mean(seq_lengths):.1f}')
ax1.axvline(np.median(seq_lengths), color='green', linestyle='--', 
           label=f'중앙값: {np.median(seq_lengths):.1f}')
ax1.set_xlabel('시퀀스 길이')
ax1.set_ylabel('빈도')
ax1.set_title('시퀀스 길이 분포')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 클래스 분포
ax2 = axes[0, 1]
label_counts = Counter(labels)
ax2.bar(['부정 (0)', '긍정 (1)'], [label_counts[0], label_counts[1]], 
       alpha=0.7, color=['red', 'green'])
ax2.set_ylabel('개수')
ax2.set_title('클래스 분포')
for i, (label, count) in enumerate(label_counts.items()):
    ax2.text(i, count + 0.5, str(count), ha='center', va='bottom')

# 3. 단어 빈도 (상위 15개)
ax3 = axes[1, 0]
word_counts = Counter()
for seq in sequences:
    word_counts.update(seq)

# 특수 토큰 제외하고 상위 15개
top_words = [(preprocessor.idx_to_word[idx], count) 
            for idx, count in word_counts.most_common(17) 
            if idx not in [0, 1]][:15]  # PAD, UNK 제외

words, counts = zip(*top_words)
y_pos = np.arange(len(words))

ax3.barh(y_pos, counts, alpha=0.7)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(words)
ax3.set_xlabel('빈도')
ax3.set_title('상위 15개 단어 빈도')
ax3.invert_yaxis()

# 4. 긍정/부정 리뷰의 평균 길이 비교
ax4 = axes[1, 1]
positive_lengths = [seq_lengths[i] for i in range(len(seq_lengths)) if labels[i] == 1]
negative_lengths = [seq_lengths[i] for i in range(len(seq_lengths)) if labels[i] == 0]

ax4.hist(positive_lengths, bins=10, alpha=0.7, label='긍정', color='green')
ax4.hist(negative_lengths, bins=10, alpha=0.7, label='부정', color='red')
ax4.set_xlabel('시퀀스 길이')
ax4.set_ylabel('빈도')
ax4.set_title('감성별 시퀀스 길이 분포')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"긍정 리뷰 평균 길이: {np.mean(positive_lengths):.2f}")
print(f"부정 리뷰 평균 길이: {np.mean(negative_lengths):.2f}")

In [ ]:
# 시퀀스 패딩
def pad_sequences(sequences, max_length=None, padding_value=0):
    """
    시퀀스들을 동일한 길이로 패딩
    """
    if max_length is None:
        max_length = max(len(seq) for seq in sequences)
    
    padded_sequences = []
    for seq in sequences:
        if len(seq) >= max_length:
            # 길면 자르기 (뒤에서부터)
            padded_seq = seq[-max_length:]
        else:
            # 짧으면 앞에 패딩 추가
            padded_seq = [padding_value] * (max_length - len(seq)) + seq
        padded_sequences.append(padded_seq)
    
    return padded_sequences

# 최대 길이 설정 (95% 분위수 사용)
max_seq_length = int(np.percentile(seq_lengths, 95))
print(f"설정된 최대 시퀀스 길이: {max_seq_length}")

# 패딩 적용
padded_sequences = pad_sequences(sequences, max_seq_length)

# 패딩 결과 확인
print(f"패딩 후 모든 시퀀스 길이: {len(padded_sequences[0])}")

# 패딩 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 패딩 전후 길이 비교
ax1.plot(seq_lengths[:20], 'b-o', label='원본 길이', alpha=0.7)
ax1.plot([max_seq_length] * 20, 'r--', label=f'패딩 후 길이 ({max_seq_length})', alpha=0.7)
ax1.set_xlabel('리뷰 인덱스')
ax1.set_ylabel('길이')
ax1.set_title('패딩 전후 길이 비교 (첫 20개)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 패딩된 시퀀스 예시
sample_padded = padded_sequences[0]
ax2.plot(sample_padded, 'go-', markersize=4)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='패딩 값 (0)')
ax2.set_xlabel('위치')
ax2.set_ylabel('토큰 ID')
ax2.set_title('패딩된 시퀀스 예시')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 패딩 통계
original_tokens = sum(seq_lengths)
padded_tokens = len(padded_sequences) * max_seq_length
padding_ratio = (padded_tokens - original_tokens) / padded_tokens * 100

print(f"\n패딩 통계:")
print(f"원본 토큰 수: {original_tokens:,}")
print(f"패딩 후 토큰 수: {padded_tokens:,}")
print(f"패딩 비율: {padding_ratio:.1f}%")

## 5. PyTorch Dataset 및 DataLoader 생성

In [ ]:
class SentimentDataset(Dataset):
    """
    감성 분석을 위한 PyTorch Dataset
    """
    def __init__(self, sequences, labels):
        self.sequences = torch.LongTensor(sequences)
        self.labels = torch.FloatTensor(labels)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

# Dataset 생성
train_dataset = SentimentDataset(X_train, y_train)
test_dataset = SentimentDataset(X_test, y_test)

# DataLoader 생성
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"훈련 데이터: {len(train_dataset)}개")
print(f"테스트 데이터: {len(test_dataset)}개")
print(f"배치 크기: {batch_size}")
print(f"훈련 배치 수: {len(train_loader)}")

# 샘플 배치 확인
sample_batch_sequences, sample_batch_labels = next(iter(train_loader))
print(f"\n샘플 배치 형태:")
print(f"시퀀스: {sample_batch_sequences.shape}")
print(f"라벨: {sample_batch_labels.shape}")

## 6. RNN 모델 구현

다양한 RNN 계열 모델을 구현하여 성능을 비교합니다.

In [ ]:
class SimpleRNNModel(nn.Module):
    """
    기본 RNN 모델
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=1):
        super(SimpleRNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        # 임베딩
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        
        # RNN
        rnn_out, hidden = self.rnn(embedded)  # (batch, seq_len, hidden_dim)
        
        # 마지막 출력 사용
        last_output = rnn_out[:, -1, :]  # (batch, hidden_dim)
        
        # 드롭아웃 및 분류
        output = self.dropout(last_output)
        output = self.fc(output)
        
        return torch.sigmoid(output)

class LSTMModel(nn.Module):
    """
    LSTM 모델
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=1):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # 마지막 출력 사용
        last_output = lstm_out[:, -1, :]
        
        output = self.dropout(last_output)
        output = self.fc(output)
        
        return torch.sigmoid(output)

class GRUModel(nn.Module):
    """
    GRU 모델
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=1):
        super(GRUModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        embedded = self.embedding(x)
        gru_out, hidden = self.gru(embedded)
        
        # 마지막 출력 사용
        last_output = gru_out[:, -1, :]
        
        output = self.dropout(last_output)
        output = self.fc(output)
        
        return torch.sigmoid(output)

class BiLSTMModel(nn.Module):
    """
    양방향 LSTM 모델
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=1):
        super(BiLSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, 
                             bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # 양방향이므로 *2
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        embedded = self.embedding(x)
        bilstm_out, (hidden, cell) = self.bilstm(embedded)
        
        # 마지막 출력 사용
        last_output = bilstm_out[:, -1, :]
        
        output = self.dropout(last_output)
        output = self.fc(output)
        
        return torch.sigmoid(output)

# 모델 파라미터 설정
vocab_size = preprocessor.vocab_size
embed_dim = 32  # 작은 데이터셋이므로 작게 설정
hidden_dim = 16

# 모델 생성
models = {
    'SimpleRNN': SimpleRNNModel(vocab_size, embed_dim, hidden_dim),
    'LSTM': LSTMModel(vocab_size, embed_dim, hidden_dim),
    'GRU': GRUModel(vocab_size, embed_dim, hidden_dim),
    'BiLSTM': BiLSTMModel(vocab_size, embed_dim, hidden_dim)
}

# 모델 정보 출력
print("모델 파라미터 수 비교:")
for name, model in models.items():
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {total_params:,} (trainable: {trainable_params:,})")

# LSTM 모델 구조 상세 출력
print(f"\nLSTM 모델 구조:")
print(models['LSTM'])

## 7. 모델 학습 및 평가

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs=20, lr=0.001):
    """
    모델을 훈련하고 평가합니다.
    """
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    train_accuracies = []
    test_accuracies = []
    
    for epoch in range(num_epochs):
        # 훈련 모드
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for sequences, labels in train_loader:
            sequences = sequences.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        
        # 테스트 평가
        test_acc = evaluate_model(model, test_loader)
        
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        test_accuracies.append(test_acc)
        
        if epoch % 5 == 0 or epoch == num_epochs - 1:
            print(f'Epoch {epoch+1}/{num_epochs}, '
                  f'Train Loss: {train_loss:.4f}, '
                  f'Train Acc: {train_acc:.4f}, '
                  f'Test Acc: {test_acc:.4f}')
    
    return {
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'test_accuracies': test_accuracies
    }

def evaluate_model(model, test_loader):
    """
    모델을 평가합니다.
    """
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for sequences, labels in test_loader:
            sequences = sequences.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            outputs = model(sequences)
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return correct / total

# 각 모델 훈련
results = {}
num_epochs = 30

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name} Model")
    print(f"{'='*50}")
    
    result = train_model(model, train_loader, test_loader, num_epochs)
    results[name] = result
    
    final_test_acc = result['test_accuracies'][-1]
    print(f"{name} 최종 테스트 정확도: {final_test_acc:.4f}")

## 8. 결과 시각화 및 비교

In [ ]:
# 학습 결과 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 훈련 손실 비교
ax1 = axes[0, 0]
for name, result in results.items():
    ax1.plot(result['train_losses'], label=name, linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('훈련 손실 비교')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 테스트 정확도 비교
ax2 = axes[0, 1]
for name, result in results.items():
    ax2.plot(result['test_accuracies'], label=name, linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy')
ax2.set_title('테스트 정확도 비교')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 최종 성능 비교
ax3 = axes[1, 0]
model_names = list(results.keys())
final_train_accs = [results[name]['train_accuracies'][-1] for name in model_names]
final_test_accs = [results[name]['test_accuracies'][-1] for name in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax3.bar(x - width/2, final_train_accs, width, label='훈련 정확도', alpha=0.7)
bars2 = ax3.bar(x + width/2, final_test_accs, width, label='테스트 정확도', alpha=0.7)

ax3.set_xlabel('모델')
ax3.set_ylabel('정확도')
ax3.set_title('최종 성능 비교')
ax3.set_xticks(x)
ax3.set_xticklabels(model_names)
ax3.legend()

# 값 표시
for bar, acc in zip(bars1, final_train_accs):
    ax3.text(bar.get_x() + bar.get_width()/2, acc + 0.01, 
            f'{acc:.3f}', ha='center', va='bottom', fontsize=9)

for bar, acc in zip(bars2, final_test_accs):
    ax3.text(bar.get_x() + bar.get_width()/2, acc + 0.01, 
            f'{acc:.3f}', ha='center', va='bottom', fontsize=9)

# 4. 과적합 분석
ax4 = axes[1, 1]
for name, result in results.items():
    train_acc = result['train_accuracies']
    test_acc = result['test_accuracies']
    overfitting = [train - test for train, test in zip(train_acc, test_acc)]
    ax4.plot(overfitting, label=name, linewidth=2)

ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Train Acc - Test Acc')
ax4.set_title('과적합 분석 (높을수록 과적합)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 최고 성능 모델 선택
best_model_name = max(results.keys(), key=lambda x: results[x]['test_accuracies'][-1])
best_accuracy = results[best_model_name]['test_accuracies'][-1]

print(f"\n🏆 최고 성능 모델: {best_model_name}")
print(f"🎯 최고 테스트 정확도: {best_accuracy:.4f}")

# 모델별 요약
print(f"\n📊 모델별 성능 요약:")
for name in model_names:
    train_acc = results[name]['train_accuracies'][-1]
    test_acc = results[name]['test_accuracies'][-1]
    overfitting = train_acc - test_acc
    print(f"{name:10}: Train {train_acc:.3f}, Test {test_acc:.3f}, "
          f"Overfitting {overfitting:.3f}")

## 9. 실제 텍스트로 예측 테스트

In [ ]:
def predict_sentiment(model, text, preprocessor, max_length):
    """
    텍스트의 감성을 예측합니다.
    """
    model.eval()
    
    # 텍스트 전처리
    sequence = preprocessor.text_to_sequence(text)
    
    # 패딩
    if len(sequence) >= max_length:
        padded_sequence = sequence[-max_length:]
    else:
        padded_sequence = [0] * (max_length - len(sequence)) + sequence
    
    # 텐서 변환
    input_tensor = torch.LongTensor([padded_sequence]).to(device)
    
    # 예측
    with torch.no_grad():
        output = model(input_tensor)
        probability = output.item()
        
        sentiment = "긍정" if probability > 0.5 else "부정"
        confidence = probability if probability > 0.5 else 1 - probability
        
        return sentiment, confidence, probability

# 테스트 문장들
test_sentences = [
    "이 영화는 정말 최고입니다",
    "너무 지루하고 재미없어요",
    "연기가 훌륭하고 스토리가 감동적입니다",
    "시간 낭비였어요 돈 아까워요",
    "그냥 그래요 보통입니다",
    "amazing movie with great acting",
    "terrible and boring film",
    "fantastic story and wonderful characters"
]

# 최고 성능 모델로 예측
best_model = models[best_model_name].to(device)

print(f"🤖 {best_model_name} 모델을 사용한 감성 분석 결과:")
print("=" * 70)

predictions = []
for i, sentence in enumerate(test_sentences):
    sentiment, confidence, probability = predict_sentiment(
        best_model, sentence, preprocessor, max_seq_length
    )
    
    predictions.append({
        'text': sentence,
        'sentiment': sentiment,
        'confidence': confidence,
        'probability': probability
    })
    
    print(f"{i+1:2d}. {sentence}")
    print(f"    → {sentiment} (신뢰도: {confidence:.1%}, 확률: {probability:.3f})")
    print()

# 예측 결과 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 감성별 신뢰도
sentiments = [p['sentiment'] for p in predictions]
confidences = [p['confidence'] for p in predictions]
colors = ['green' if s == '긍정' else 'red' for s in sentiments]

bars = ax1.bar(range(len(predictions)), confidences, color=colors, alpha=0.7)
ax1.set_xlabel('문장 번호')
ax1.set_ylabel('신뢰도')
ax1.set_title('문장별 예측 신뢰도')
ax1.set_xticks(range(len(predictions)))
ax1.set_xticklabels([f'{i+1}' for i in range(len(predictions))])

# 범례 추가
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', alpha=0.7, label='긍정'),
                  Patch(facecolor='red', alpha=0.7, label='부정')]
ax1.legend(handles=legend_elements)

# 확률 분포
probabilities = [p['probability'] for p in predictions]
ax2.hist(probabilities, bins=10, alpha=0.7, edgecolor='black')
ax2.axvline(0.5, color='red', linestyle='--', linewidth=2, label='결정 경계')
ax2.set_xlabel('예측 확률')
ax2.set_ylabel('빈도')
ax2.set_title('예측 확률 분포')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 감성별 통계
positive_count = sum(1 for s in sentiments if s == '긍정')
negative_count = len(sentiments) - positive_count
avg_confidence = np.mean(confidences)

print(f"📈 예측 통계:")
print(f"긍정 예측: {positive_count}개 ({positive_count/len(sentiments)*100:.1f}%)")
print(f"부정 예측: {negative_count}개 ({negative_count/len(sentiments)*100:.1f}%)")
print(f"평균 신뢰도: {avg_confidence:.1%}")

## 10. RNN vs LSTM vs GRU 상세 비교

In [ ]:
# RNN 계열 모델들의 상세 비교
def create_model_comparison():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. 모델 복잡도 vs 성능
    ax1 = axes[0, 0]
    
    # 파라미터 수 계산
    param_counts = []
    performance_scores = []
    model_names_plot = []
    
    for name, model in models.items():
        params = sum(p.numel() for p in model.parameters())
        performance = results[name]['test_accuracies'][-1]
        param_counts.append(params)
        performance_scores.append(performance)
        model_names_plot.append(name)
    
    colors = ['red', 'blue', 'green', 'orange']
    for i, (name, params, perf) in enumerate(zip(model_names_plot, param_counts, performance_scores)):
        ax1.scatter(params, perf, s=200, c=colors[i], alpha=0.7, label=name)
        ax1.annotate(name, (params, perf), xytext=(5, 5), textcoords='offset points')
    
    ax1.set_xlabel('모델 파라미터 수')
    ax1.set_ylabel('테스트 정확도')
    ax1.set_title('모델 복잡도 vs 성능')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # 2. 학습 속도 비교 (에폭당 수렴 속도)
    ax2 = axes[0, 1]
    
    # 80% 정확도에 도달하는 에폭 계산
    epochs_to_80 = []
    for name in model_names_plot:
        test_accs = results[name]['test_accuracies']
        epoch_80 = next((i for i, acc in enumerate(test_accs) if acc >= 0.8), len(test_accs))
        epochs_to_80.append(epoch_80)
    
    bars = ax2.bar(model_names_plot, epochs_to_80, color=colors, alpha=0.7)
    ax2.set_ylabel('80% 정확도 도달 에폭')
    ax2.set_title('모델별 수렴 속도')
    ax2.set_xticklabels(model_names_plot, rotation=45)
    
    for bar, epoch in zip(bars, epochs_to_80):
        ax2.text(bar.get_x() + bar.get_width()/2, epoch + 0.5, 
                str(epoch), ha='center', va='bottom')
    
    # 3. 과적합 경향 비교
    ax3 = axes[1, 0]
    
    final_overfitting = []
    for name in model_names_plot:
        train_acc = results[name]['train_accuracies'][-1]
        test_acc = results[name]['test_accuracies'][-1]
        overfitting = train_acc - test_acc
        final_overfitting.append(overfitting)
    
    bars = ax3.bar(model_names_plot, final_overfitting, color=colors, alpha=0.7)
    ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax3.set_ylabel('과적합 정도 (Train - Test Acc)')
    ax3.set_title('모델별 과적합 경향')
    ax3.set_xticklabels(model_names_plot, rotation=45)
    
    for bar, overfit in zip(bars, final_overfitting):
        ax3.text(bar.get_x() + bar.get_width()/2, overfit + 0.01, 
                f'{overfit:.3f}', ha='center', va='bottom')
    
    # 4. 모델 특성 비교 테이블
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # 특성 비교 데이터
    characteristics = {
        'SimpleRNN': ['빠른 학습', '기울기 소실', '단순 구조', '기본 성능'],
        'LSTM': ['복잡한 구조', '장기 기억', '높은 성능', '많은 파라미터'],
        'GRU': ['LSTM 간소화', '적은 파라미터', '좋은 성능', '빠른 학습'],
        'BiLSTM': ['양방향 처리', '최고 성능', '많은 연산', '문맥 이해']
    }
    
    y_pos = 0.9
    ax4.text(0.5, 0.95, '모델별 주요 특성', ha='center', va='top', 
            fontsize=14, weight='bold', transform=ax4.transAxes)
    
    for model, features in characteristics.items():
        ax4.text(0.05, y_pos, f'{model}:', weight='bold', 
                transform=ax4.transAxes, fontsize=12)
        
        for i, feature in enumerate(features):
            ax4.text(0.1, y_pos - 0.04 * (i + 1), f'• {feature}', 
                    transform=ax4.transAxes, fontsize=10)
        
        y_pos -= 0.22
    
    plt.tight_layout()
    plt.show()

create_model_comparison()

# 모델별 권장 사용 시나리오
print("\n🎯 모델별 권장 사용 시나리오:")
print("="*50)
print("📌 SimpleRNN: 간단한 시퀀스 분류, 빠른 프로토타이핑")
print("📌 LSTM: 긴 시퀀스, 복잡한 의존성, 높은 정확도 필요")
print("📌 GRU: LSTM과 비슷한 성능, 더 빠른 학습 원할 때")
print("📌 BiLSTM: 문맥을 양방향으로 고려해야 하는 경우")

## 11. 학습 요약 및 다음 단계

In [ ]:
# 학습 내용 요약
def create_learning_summary():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. RNN 발전 과정
    ax1 = axes[0, 0]
    
    timeline = ['기본 RNN\n(1986)', 'LSTM\n(1997)', 'GRU\n(2014)', 'Transformer\n(2017)']
    years = [1986, 1997, 2014, 2017]
    improvements = [1, 3, 3.5, 5]  # 상대적 성능 점수
    
    ax1.plot(years, improvements, 'o-', linewidth=3, markersize=10)
    ax1.set_xlabel('연도')
    ax1.set_ylabel('상대적 성능')
    ax1.set_title('RNN 기술의 발전')
    ax1.grid(True, alpha=0.3)
    
    for i, (year, score, label) in enumerate(zip(years, improvements, timeline)):
        ax1.annotate(label, (year, score), xytext=(0, 20), 
                    textcoords='offset points', ha='center', 
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7))
    
    # 2. 주요 개념 중요도
    ax2 = axes[0, 1]
    
    concepts = ['시퀀스\n처리', '기울기\n소실', 'LSTM\n게이트', '양방향\n처리', '어텐션\n메커니즘']
    importance = [5, 4, 5, 3, 4]
    colors = plt.cm.viridis(np.linspace(0, 1, len(concepts)))
    
    bars = ax2.bar(concepts, importance, color=colors, alpha=0.8)
    ax2.set_ylabel('중요도 (1-5)')
    ax2.set_title('핵심 개념 중요도')
    ax2.set_ylim(0, 6)
    
    for bar, imp in zip(bars, importance):
        ax2.text(bar.get_x() + bar.get_width()/2, imp + 0.1, 
                str(imp), ha='center', va='bottom', fontweight='bold')
    
    # 3. 실제 응용 분야
    ax3 = axes[1, 0]
    
    applications = ['감성 분석', '기계 번역', '음성 인식', '주가 예측', '챗봇']
    usage_percentage = [25, 20, 20, 15, 20]
    
    wedges, texts, autotexts = ax3.pie(usage_percentage, labels=applications, 
                                      autopct='%1.1f%%', startangle=90,
                                      colors=colors)
    ax3.set_title('RNN/LSTM 실제 활용 분야')
    
    # 4. 학습 체크리스트
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    checklist = [
        "✅ RNN의 기본 개념과 시퀀스 처리 이해",
        "✅ LSTM과 GRU의 구조와 차이점 파악",
        "✅ 텍스트 전처리와 토큰화 과정 이해",
        "✅ PyTorch를 활용한 RNN 모델 구현",
        "✅ 다양한 RNN 모델 성능 비교",
        "✅ 실제 데이터로 감성 분석 수행",
        "✅ 모델 평가 및 결과 해석",
        "✅ 과적합 방지 및 모델 최적화"
    ]
    
    ax4.text(0.05, 0.95, '🎓 학습 체크리스트', 
            fontsize=16, weight='bold', transform=ax4.transAxes)
    
    for i, item in enumerate(checklist):
        ax4.text(0.05, 0.85 - i*0.1, item, fontsize=12, 
                transform=ax4.transAxes)
    
    plt.tight_layout()
    plt.show()

create_learning_summary()

print("🎉 RNN/LSTM 기초 학습 완료!")
print("\n📚 오늘 학습한 내용:")
print("1️⃣  순환 신경망(RNN)의 기본 원리와 한계")
print("2️⃣  LSTM과 GRU의 구조와 장점")
print("3️⃣  텍스트 전처리 및 시퀀스 패딩")
print("4️⃣  PyTorch를 활용한 RNN 모델 구현")
print("5️⃣  실제 데이터로 감성 분석 실습")
print("6️⃣  모델 성능 비교 및 평가")

print("\n🚀 다음 단계 추천:")
print("📖 Transformer와 Attention 메커니즘 학습")
print("📖 BERT, GPT 같은 사전 훈련 모델 활용")
print("📖 더 큰 데이터셋으로 실습 (IMDB, 영화 리뷰 등)")
print("📖 한국어 자연어 처리 실습")
print("📖 시계열 데이터 예측 문제 도전")

print("\n💡 실습 팁:")
print("• 다양한 하이퍼파라미터로 실험해보세요")
print("• 더 많은 데이터로 모델 성능을 향상시켜보세요")
print("• 드롭아웃, 정규화 등으로 과적합을 방지해보세요")
print("• 실제 프로젝트에 적용해보세요")

if device.type != 'cpu':
    print(f"\n🔥 GPU({device}) 가속을 활용하여 더 큰 모델도 시도해보세요!")
else:
    print("\n💻 CPU로도 충분히 학습할 수 있지만, GPU가 있다면 더 큰 모델을 시도해보세요!")